In [ ]:
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import seaborn as sns
import numpy as np
import sys
import os

from data_pipeline import get_full_sequence_pipeline

CONFIG = {
    "base": dict(c1=32,c2=64,h=128,layers=2),
    "small": dict(c1=16,c2=32,h=64,layers=2),
    "tiny": dict(c1=8,c2=16,h=32,layers=1)
}


def run_stmae(freeze_encoder: bool, mode: int, size="base", window=None, batch_size=512):
  # load data
  (loaders, scaler, le) = get_full_sequence_pipeline(mode, window=window, batch_size=batch_size)

  train_loader, val_loader, test_loader, n_features, n_classes = loaders


  # masking function
  def mask_features(x, mask_ratio=0.3):
    mask = torch.rand(x.shape[0], x.shape[1], device=x.device) < mask_ratio
    mask = mask.unsqueeze(-1) # should still unsqueeze here since this is the process of extracting shape of data per dimensions
    x_masked = x * (~mask).float()
    return x_masked, mask

  class Encoder(nn.Module):
    def __init__(self, n_features, hidden_dim):
      super().__init__()
      cfg = CONFIG[size]

      self.cnn = nn.Sequential(
          nn.Conv1d(n_features, cfg["c1"], kernel_size=3, padding=1),
          nn.ReLU(),
          nn.Conv1d(cfg["c1"], cfg["c2"], kernel_size=3, padding=1),
          nn.ReLU()
      )

      self.lstm = nn.LSTM(
          input_size=cfg["c2"],
          hidden_size=hidden_dim,
          num_layers=cfg["layers"],
          batch_first=True
      )

    def forward(self, x):
      x = x.permute(0, 2, 1)
      x = self.cnn(x)
      x = x.permute(0, 2, 1)
      clean_temporal = x.clone()
      x_masked, mask = mask_features(x)
      seq_out, (h_n, c_n) = self.lstm(x_masked)
      return seq_out, clean_temporal, mask

  class Decoder(nn.Module):
    def __init__(self, n_features, hidden_dim):
      super().__init__()
      cfg = CONFIG[size]

      self.temporal_dim = cfg["c2"]

      self.net = nn.Sequential(
          nn.Linear(hidden_dim, 2*hidden_dim),
          nn.ReLU(),
          nn.Linear(2*hidden_dim, self.temporal_dim)
      )

    def forward(self, x):
      recon = self.net(x)
      return recon

  # device
  device = "cuda" if torch.cuda.is_available() else "cpu"
  hidden_dim = CONFIG[size]["h"]

  # model
  class ST_MAE(nn.Module):
    def __init__(self, n_features, hidden_dim):
      super().__init__()
      self.encoder = Encoder(n_features, hidden_dim)
      self.decoder = Decoder(n_features, hidden_dim)

    def forward(self, x):
      seq_out, clean_temporal, mask = self.encoder(x)
      x_recon = self.decoder(seq_out)
      return x_recon, clean_temporal, mask

  # training loop
  model = ST_MAE(n_features, hidden_dim).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  def stmae_epoch(loader, train=True):
    if train:
      model.train()
    else:
      model.eval()

    total_err = 0
    total_mask = 0

    with torch.set_grad_enabled(train):
      for X, _ in loader:
        X = X.to(device)
        X_recon, clean_temporal, mask = model(X)

        err = (X_recon - clean_temporal) ** 2
        masked_err_sum = (err * mask).sum()
        masked_count = mask.sum()

        loss = masked_err_sum / (masked_count + 1e-8)

        if train:
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

        total_err += masked_err_sum.item()
        total_mask += masked_count.item()

    return total_err / (total_mask + 1e-8)


  for epoch in range(10):
    train_recon = stmae_epoch(train_loader)
    val_recon = stmae_epoch(val_loader, train=False)

  # after training discard decoder
  encoder = model.encoder
  encoder.eval()

  # attach classification head
  class ST_MAE_Classifier(nn.Module):
    def __init__(self, trained_encoder, hidden_dim, n_classes):
      super().__init__()
      self.encoder = trained_encoder
      self.dropout = nn.Dropout(0.3)
      self.attn = nn.Linear(hidden_dim, 1)
      self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
      seq_out, _, _ = self.encoder(x)

      weights = torch.softmax(self.attn(seq_out), dim=1)
      context = torch.sum(weights * seq_out, dim=1)
      context = self.dropout(context)

      logits = self.fc(context)
      return logits

  criterion = nn.CrossEntropyLoss()
  classifier = ST_MAE_Classifier(encoder, hidden_dim, n_classes).to(device)
  if freeze_encoder:
    for param in classifier.encoder.parameters():
      param.requires_grad = False
  optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

  def classify_epoch(loader, train=True):
    if train:
      classifier.train()
      if freeze_encoder:
        classifier.encoder.eval()
    else:
      classifier.eval()

    total_loss = 0
    n = 0

    all_preds = []
    all_true = []

    with torch.set_grad_enabled(train):
      for X, y in loader:
        X = X.to(device)
        y = y.to(device)
        logits = classifier(X)
        loss = criterion(logits, y)

        if train:
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

        total_loss += loss.item() * y.size(0)
        n += y.size(0)

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.append(preds)
        all_true.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_true = np.concatenate(all_true)

    avg_loss = total_loss / n
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    acc = accuracy_score(all_true, all_preds)

    return{
        "loss": avg_loss,
        "macro_f1": macro_f1,
        "acc": acc,
        "preds": all_preds,
        "true": all_true
    }

  train_losses = []
  val_losses = []
  train_f1s = []
  val_f1s = []
  train_accs = []
  val_accs = []

  for epoch in range(10):
    train_epoch = classify_epoch(train_loader)
    val_epoch = classify_epoch(val_loader, train=False)
    train_losses.append(train_epoch["loss"])
    val_losses.append(val_epoch["loss"])
    train_f1s.append(train_epoch["macro_f1"])
    val_f1s.append(val_epoch["macro_f1"])
    train_accs.append(train_epoch["acc"])
    val_accs.append(val_epoch["acc"])
    train_loss = train_losses[-1]
    val_loss = val_losses[-1]
    train_f1 = train_f1s[-1]
    val_f1 = val_f1s[-1]
    train_acc = train_accs[-1]
    val_acc = val_accs[-1]

  test_epoch = classify_epoch(test_loader, train=False)
  test_loss = test_epoch["loss"]
  test_f1 = test_epoch["macro_f1"]
  test_preds = test_epoch["preds"]
  test_trues = test_epoch["true"]

  accuracy = accuracy_score(test_trues, test_preds)
  precision, recall, f1, _ = precision_recall_fscore_support(test_trues, test_preds, average='macro', zero_division=0)
  cm = confusion_matrix(test_trues, test_preds)

  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'train_loss': train_losses,
      'val_loss': val_losses,
      'confusion_matrix': cm,
      'history': val_losses
  }


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive

Epoch 1:
Train reconstruction error: 289989571789.54553
Validation reconstruction error: 140.53178221779106
Epoch 2:
Train reconstruction error: 25.995533635153024
Validation reconstruction error: 4.437062268700782
Epoch 3:
Train reconstruction error: 25.366360854842473
Validation reconstruction error: 2.1381131643264575
Epoch 4:
Train reconstruction error: 1.6060102559885643
Validation reconstruction error: 1.1673406661721686
Epoch 5:
Train reconstruction error: 0.8609793357888206
Validation reconstruction error: 0.6040065986499548
Epoch 6:
Train reconstruction error: 0.43160729402217335
Validation reconstruction error: 0.2929534430050527
Epoch 7:
Train reconstruction error: 0.21114113822780847
Validation reconstruction error: 0.1503949910342468
Epoch 8:
Train reconstruction error: 0.11981752108885653
Validation reconstruction error: 0.09780060863312752
Epoch 9:
Train reconstruction error: 0.08421438825483794
Validation reconstruction error: 0.07251302568797971
Epoch 10:
Train reconst